In [3]:
import GtoTmodel
import torch

In [4]:
torch.manual_seed(1337)
torch.cuda.manual_seed(1337)
embed_dim = 16  # Embedding dimension
num_heads = 4  # Number of attention heads
num_layers = 2  # Number of transformer layers
dropout = 0.1  # Dropout rate

#graph_colomns=5
num_components=5
batch_size = 16  # Batch size

graph_input_dim = 310  # Number of colomns in the graph
text_vocab_size = 894  # Vocabulary size for text

In [5]:
# Check if GPU is available and set the device
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("Using CPU")

Using GPU: NVIDIA GeForce RTX 3090


In [ ]:
model = GtoTmodel.GraphToTextTransformer(
    graph_input_dim,
    text_vocab_size, 
    embed_dim, 
    num_heads, 
    num_layers, 
    dropout)

model = model.to(device)

/home/nithira/circuits_gen/.venv/lib/python3.10/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [7]:
# Define the file path to load the model and hyperparameters
load_path = "model_checkpoint.pth"

# Load the checkpoint
checkpoint = torch.load(load_path)

# Restore the model state and hyperparameters
model.load_state_dict(checkpoint['model_state_dict'])

# Restore hyperparameters if needed
embed_dim = checkpoint['embed_dim']
num_heads = checkpoint['num_heads']
num_layers = checkpoint['num_layers']
dropout = checkpoint['dropout']
text_vocab_size = checkpoint['text_vocab_size']
graph_input_dim = checkpoint['graph_input_dim']

print(f"Model and optimizer state loaded from {load_path}")

Model and optimizer state loaded from model_checkpoint.pth


In [19]:
model.eval()
    

# Example: Test the model with a single sample
# Generate random test data
test_graph = torch.rand((16, graph_input_dim), device=device).unsqueeze(0)  # Random graph input
test_text = torch.randint(0, text_vocab_size, (1, 10), device=device)  # Random text input
print("Graph Input Shape:", test_graph.shape)
print("Text Input Shape:", test_text.shape)

# test_graph = graph_val[0].unsqueeze(0)
# test_text = text_val[:16].unsqueeze(0) 
with torch.no_grad(): 

        if test_text.dim() == 1:
            test_text = test_text.unsqueeze(0)

        if test_graph.dim() == 1:
            test_graph = test_graph.unsqueeze(0)

        # Forward pass through the model
        output = model(test_graph, test_text[:, :-1])  # Exclude the last token for input
        print("Output Shape:", output.shape)
        predicted_tokens = torch.argmax(output, dim=-1)



print("Predicted Tokens Shape:", predicted_tokens.shape)
print("Predicted Tokens:", predicted_tokens)
print("Actual Tokens:", test_text[:, 1:])  # Exclude the first token for comparison

Graph Input Shape: torch.Size([1, 16, 310])
Text Input Shape: torch.Size([1, 10])
Output Shape: torch.Size([1, 9, 894])
Predicted Tokens Shape: torch.Size([1, 9])
Predicted Tokens: tensor([[273, 273, 394, 644, 300, 273, 684, 490, 783]], device='cuda:0')
Actual Tokens: tensor([[529, 445, 800, 659,  27, 865, 145, 243, 873]], device='cuda:0')
